In [1]:
from langchain_community.document_loaders import UnstructuredURLLoader

/Users/anikparui/Desktop/Project/RAG/rag_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
url = ["https://starsunfolded.com/mahendra-singh-dhoni/"]

In [3]:
loader = UnstructuredURLLoader(urls = url)

In [4]:
data = loader.load()

In [5]:
data

[Document(metadata={'source': 'https://starsunfolded.com/mahendra-singh-dhoni/'}, page_content='Menu\n\nStarsUnfolded\n\nMahendra Singh Dhoni Height, Age, Girlfriend, Wife, Children, Family, Biography\n\nQuick Info→\n\nHometown: Ranchi, Jharkhand\n\nMarriage Date: 4 July 2010\n\nAge: 44 Years\n\nMS Dhoni\n\nBio/Wiki Full Name Mahendra Singh Dhoni Real Name Mahendra Singh Dhauni [1] Business Standard Nickname Mahi Names Earned MSD, MS, Captain Cool Profession Cricketer (Wicket-keeper) Physical Stats & More Height (approx.) 5\' 9" (175 cm) Weight (approx.) 75 Kg (165 lbs) Body Measurements (approx.) - Chest: 42 inches - Waist: 32 inches - Biceps: 14 inches Eye Colour Dark Brown Hair Colour Black Cricket International Debut ODI- 23 December 2004 against Bangladesh at Chittagong Test- 2 December 2005 against Sri Lanka at Chennai T20 - 1 December 2006 against South Africa at Johannesburg Last Match ODI - 9 July 2019 against New Zealand at Emirates Old Trafford Test - 26 December 2014 agains

In [6]:
print("The length of the data = ", len(data))

The length of the data =  1


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
#split the data
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100)
docs = text_splitter.split_documents(data)

In [9]:
print("The number of documents = ", len(docs))

The number of documents =  53


In [10]:
#load ing huggingface embedding class
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings()

vector = embeddings.embed_query("hello")

/var/folders/49/dlfryms541x6nvjcz5yz0hh80000gn/T/ipykernel_28397/1865281471.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/var/folders/49/dlfryms541x6nvjcz5yz0hh80000gn/T/ipykernel_28397/1865281471.py:3: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 5799.56it/s]


In [11]:
print(len(vector))

768


In [12]:
#creating a vector database to store all the values of vector embeddings
# the vector database is named as vectorstore

from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(docs, embedding = embeddings)

In [13]:
# we want to use this vector database as retriver
# the task of retriver is to use those documents that matches the user's query

retriever = vectorstore.as_retriever(search_type = "similarity", search_kwargs = {"k" : 10})
retrieved_docs = retriever.invoke("MS Dhoni")

In [14]:
# printing the length of the retrieved document

print("Length of the retrived document = ", len(retrieved_docs))

Length of the retrived document =  10


In [16]:
# print the content of any document

print(retrieved_docs[7].page_content)

Menu

StarsUnfolded

Mahendra Singh Dhoni Height, Age, Girlfriend, Wife, Children, Family, Biography

Quick Info→

Hometown: Ranchi, Jharkhand

Marriage Date: 4 July 2010

Age: 44 Years

MS Dhoni


In [20]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

# Use Groq API key to load the Groq llm
api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key = api_key, model_name = "llama-3.3-70b-versatile")

prompt_template = """
You are a factual assistant.

Answer the question ONLY using the context below.
- Do NOT use prior knowledge
- Do NOT refuse unless context is empty
- If answer exists in context, give it directly

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template
)

# Chain
llm_chain = prompt | llm | StrOutputParser()

In [21]:
# As we are creating RAG application so we have to create a RAG chain
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | llm_chain
)

In [22]:
#Input question
question = "who is the wife of Dhoni?"

#answer
response = rag_chain.invoke(question)
print(response)

Sakshi Dhoni
